In [1]:
# ============================================================
# GLOBAL PRICE WATCH
# TOPIC 07 — LIGHTGBM MODEL
# Cell 1: Imports
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb

warnings.filterwarnings("ignore")

print("=" * 60)
print("LIGHTGBM MODELING")
print("=" * 60)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("LightGBM:", lgb.__version__)

LIGHTGBM MODELING
NumPy: 2.4.6
Pandas: 3.0.5
LightGBM: 4.7.0


In [4]:
# ============================================================
# Cell 2: Load leakage-safe features + cluster information
# ============================================================

import os
import pandas as pd

FEATURE_FILE = "inflation_feature_engineered_leakage_safe.csv"
CLUSTER_FILE = "final_country_clusters.csv"

# ------------------------------------------------------------
# Check files
# ------------------------------------------------------------

if not os.path.exists(FEATURE_FILE):
    raise FileNotFoundError(
        f"Missing file: {FEATURE_FILE}"
    )

if not os.path.exists(CLUSTER_FILE):
    raise FileNotFoundError(
        f"Missing file: {CLUSTER_FILE}"
    )

# ------------------------------------------------------------
# Load datasets
# ------------------------------------------------------------

modeling_df = pd.read_csv(FEATURE_FILE)
cluster_df = pd.read_csv(CLUSTER_FILE)

print("=" * 60)
print("DATASETS LOADED")
print("=" * 60)

print("Feature dataset:", modeling_df.shape)
print("Cluster dataset:", cluster_df.shape)

print("\nCluster columns:")
print(cluster_df.columns.tolist())

# ------------------------------------------------------------
# Keep only required cluster information
# ------------------------------------------------------------

cluster_info = cluster_df[
    ["Country Name", "Country Code", "Cluster"]
].copy()

# ------------------------------------------------------------
# Merge cluster labels
# ------------------------------------------------------------

modeling_df = modeling_df.merge(
    cluster_info,
    on=["Country Name", "Country Code"],
    how="left"
)

print("\n" + "=" * 60)
print("CLUSTER INFORMATION MERGED")
print("=" * 60)

print("Final shape:", modeling_df.shape)

print("\nCluster missing values:")
print(modeling_df["Cluster"].isna().sum())

print("\nCluster counts:")
print(modeling_df["Cluster"].value_counts())

display(
    modeling_df[
        ["Country Name", "Country Code", "Cluster"]
    ].head(10)
)

DATASETS LOADED
Feature dataset: (193, 145)
Cluster dataset: (193, 3)

Cluster columns:
['Country Name', 'Country Code', 'Cluster']

CLUSTER INFORMATION MERGED
Final shape: (193, 146)

Cluster missing values:
0

Cluster counts:
Cluster
1    126
0     67
Name: count, dtype: int64


,Country Name,Country Code,Cluster
0,Aruba,ABW,0
1,Afghanistan,AFG,0
2,Angola,AGO,1
3,Albania,ALB,0
4,United Arab Emirates,ARE,0
5,Argentina,ARG,1
6,Armenia,ARM,0
7,Antigua and Barbuda,ATG,1
8,Australia,AUS,1
9,Austria,AUT,1


In [5]:
# ============================================================
# Cell 3: Dataset validation
# ============================================================

print("=" * 60)
print("DATASET VALIDATION")
print("=" * 60)

print("Shape:", modeling_df.shape)

print("\nMissing values:")
missing = modeling_df.isnull().sum()
missing = missing[missing > 0]

if len(missing) == 0:
    print("✓ No missing values")
else:
    print(missing)

print("\nColumns:")
print("Total columns:", len(modeling_df.columns))

print("\nRequired columns:")
required_columns = ["Country Name", "Country Code", "Cluster"]

for col in required_columns:
    if col in modeling_df.columns:
        print(f"✓ {col}")
    else:
        print(f"✗ {col} MISSING")

DATASET VALIDATION
Shape: (193, 146)

Missing values:
✓ No missing values

Columns:
Total columns: 146

Required columns:
✓ Country Name
✓ Country Code
✓ Cluster


In [6]:
# ============================================================
# Cell 4: Identify year columns
# ============================================================

year_columns = [
    col for col in modeling_df.columns
    if str(col).isdigit() and 1960 <= int(col) <= 2025
]

year_columns = sorted(year_columns, key=int)

print("=" * 60)
print("YEAR COLUMNS")
print("=" * 60)

print("Number of year columns:", len(year_columns))
print("First year:", year_columns[0])
print("Last year:", year_columns[-1])

print("\nYears:")
print(year_columns)

YEAR COLUMNS
Number of year columns: 66
First year: 1960
Last year: 2025

Years:
['1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [7]:
# ============================================================
# Cell 5: Create LightGBM supervised dataset
# ============================================================

WINDOW_SIZE = 10

samples = []

for _, row in modeling_df.iterrows():

    country_name = row["Country Name"]
    country_code = row["Country Code"]
    cluster = row["Cluster"]

    values = row[year_columns].values.astype(float)

    for i in range(WINDOW_SIZE, len(year_columns)):

        target_year = int(year_columns[i])

        history = values[i - WINDOW_SIZE:i]
        target = values[i]

        if np.isnan(history).any() or np.isnan(target):
            continue

        sample = {
            "Country Name": country_name,
            "Country Code": country_code,
            "Cluster": cluster,
            "Target_Year": target_year
        }

        # Historical inflation values
        for lag in range(WINDOW_SIZE):
            sample[f"Inflation_Lag_{WINDOW_SIZE-lag}"] = history[lag]

        sample["Target"] = target

        samples.append(sample)

lightgbm_df = pd.DataFrame(samples)

print("=" * 60)
print("LIGHTGBM SUPERVISED DATASET")
print("=" * 60)

print("Shape:", lightgbm_df.shape)
print("Countries:", lightgbm_df["Country Code"].nunique())

print("\nTarget years:")
print(
    lightgbm_df["Target_Year"].min(),
    "-",
    lightgbm_df["Target_Year"].max()
)

display(lightgbm_df.head())

LIGHTGBM SUPERVISED DATASET
Shape: (10808, 15)
Countries: 193

Target years:
1970 - 2025


,Country Name,Country Code,Cluster,Target_Year,Inflation_Lag_10,Inflation_Lag_9,Inflation_Lag_8,Inflation_Lag_7,Inflation_Lag_6,Inflation_Lag_5,Inflation_Lag_4,Inflation_Lag_3,Inflation_Lag_2,Inflation_Lag_1,Target
0,Aruba,ABW,0,1970,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258
1,Aruba,ABW,0,1971,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258
2,Aruba,ABW,0,1972,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258
3,Aruba,ABW,0,1973,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258
4,Aruba,ABW,0,1974,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258


In [8]:
# ============================================================
# Cell 6: Check supervised dataset
# ============================================================

print("=" * 60)
print("SUPERVISED DATA CHECK")
print("=" * 60)

print("Missing values:", lightgbm_df.isnull().sum().sum())

print("\nTarget statistics:")
print(lightgbm_df["Target"].describe())

print("\nSamples by target year:")
print(
    lightgbm_df["Target_Year"]
    .value_counts()
    .sort_index()
    .head(20)
)

SUPERVISED DATA CHECK
Missing values: 0

Target statistics:
count    10808.000000
mean        55.850202
std        417.161223
min        -17.640424
25%          2.216405
50%          5.628645
75%         11.960781
max      23773.131774
Name: Target, dtype: float64

Samples by target year:
Target_Year
1970    193
1971    193
1972    193
1973    193
1974    193
1975    193
1976    193
1977    193
1978    193
1979    193
1980    193
1981    193
1982    193
1983    193
1984    193
1985    193
1986    193
1987    193
1988    193
1989    193
Name: count, dtype: int64


In [9]:
# ============================================================
# Cell 7: Time-based train / validation / test split
# ============================================================

TRAIN_END_YEAR = 2019
VALIDATION_END_YEAR = 2022

train_df = lightgbm_df[
    lightgbm_df["Target_Year"] <= TRAIN_END_YEAR
].copy()

val_df = lightgbm_df[
    (lightgbm_df["Target_Year"] > TRAIN_END_YEAR) &
    (lightgbm_df["Target_Year"] <= VALIDATION_END_YEAR)
].copy()

test_df = lightgbm_df[
    lightgbm_df["Target_Year"] > VALIDATION_END_YEAR
].copy()

print("=" * 60)
print("TIME-BASED DATA SPLIT")
print("=" * 60)

print("TRAIN:")
print(train_df.shape)

print("\nVALIDATION:")
print(val_df.shape)

print("\nTEST:")
print(test_df.shape)

print("\nYear ranges:")

print(
    "Train:",
    train_df["Target_Year"].min(),
    "-",
    train_df["Target_Year"].max()
)

print(
    "Validation:",
    val_df["Target_Year"].min(),
    "-",
    val_df["Target_Year"].max()
)

print(
    "Test:",
    test_df["Target_Year"].min(),
    "-",
    test_df["Target_Year"].max()
)

TIME-BASED DATA SPLIT
TRAIN:
(9650, 15)

VALIDATION:
(579, 15)

TEST:
(579, 15)

Year ranges:
Train: 1970 - 2019
Validation: 2020 - 2022
Test: 2023 - 2025


In [10]:
# ============================================================
# Cell 8: Define LightGBM features
# ============================================================

feature_columns = [
    col for col in lightgbm_df.columns
    if col.startswith("Inflation_Lag_")
]

# Add cluster information
feature_columns.append("Cluster")

print("=" * 60)
print("LIGHTGBM FEATURES")
print("=" * 60)

print("Number of features:", len(feature_columns))

for i, feature in enumerate(feature_columns, start=1):
    print(f"{i:2d}. {feature}")

LIGHTGBM FEATURES
Number of features: 11
 1. Inflation_Lag_10
 2. Inflation_Lag_9
 3. Inflation_Lag_8
 4. Inflation_Lag_7
 5. Inflation_Lag_6
 6. Inflation_Lag_5
 7. Inflation_Lag_4
 8. Inflation_Lag_3
 9. Inflation_Lag_2
10. Inflation_Lag_1
11. Cluster


In [11]:
# ============================================================
# Cell 9: Prepare train / validation / test arrays
# ============================================================

X_train = train_df[feature_columns].copy()
y_train = train_df["Target"].copy()

X_val = val_df[feature_columns].copy()
y_val = val_df["Target"].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df["Target"].copy()

print("=" * 60)
print("LIGHTGBM INPUT SHAPES")
print("=" * 60)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

LIGHTGBM INPUT SHAPES
X_train: (9650, 11)
y_train: (9650,)
X_val  : (579, 11)
y_val  : (579,)
X_test : (579, 11)
y_test : (579,)


In [12]:
# ============================================================
# Cell 10: Train LightGBM
# ============================================================

lgbm_model = lgb.LGBMRegressor(
    objective="huber",
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1
)

print("=" * 60)
print("TRAINING LIGHTGBM")
print("=" * 60)

lgbm_model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=["train", "validation"],
    callbacks=[
        lgb.early_stopping(80),
        lgb.log_evaluation(50)
    ]
)

print("\nLightGBM training completed.")
print("Best iteration:", lgbm_model.best_iteration_)

TRAINING LIGHTGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000921 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2545
[LightGBM] [Info] Number of data points in the train set: 9650, number of used features: 11
[LightGBM] [Info] Start training from score 61.307111
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 80 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

In [13]:
# ============================================================
# Cell 11: LightGBM evaluation
# ============================================================

def calculate_lgbm_metrics(model, X, y, dataset_name):

    predictions = model.predict(X)

    mae = mean_absolute_error(y, predictions)
    rmse = np.sqrt(mean_squared_error(y, predictions))
    r2 = r2_score(y, predictions)

    print(f"\n{dataset_name}")
    print("-" * 40)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    return predictions, mae, rmse, r2


print("=" * 60)
print("LIGHTGBM PERFORMANCE")
print("=" * 60)

lgb_train_pred, lgb_train_mae, lgb_train_rmse, lgb_train_r2 = \
    calculate_lgbm_metrics(
        lgbm_model,
        X_train,
        y_train,
        "TRAIN"
    )

lgb_val_pred, lgb_val_mae, lgb_val_rmse, lgb_val_r2 = \
    calculate_lgbm_metrics(
        lgbm_model,
        X_val,
        y_val,
        "VALIDATION"
    )

lgb_test_pred, lgb_test_mae, lgb_test_rmse, lgb_test_r2 = \
    calculate_lgbm_metrics(
        lgbm_model,
        X_test,
        y_test,
        "TEST"
    )

LIGHTGBM PERFORMANCE

TRAIN
----------------------------------------
MAE  : 73.5530
RMSE : 436.0800
R²   : 0.0222

VALIDATION
----------------------------------------
MAE  : 31.3407
RMSE : 40.5589
R²   : -0.2551

TEST
----------------------------------------
MAE  : 30.8304
RMSE : 34.3361
R²   : -0.4661


In [14]:
# ============================================================
# Cell 12: GRU vs LightGBM
# ============================================================

gru_mae = 7.0178
gru_rmse = 20.7851
gru_r2 = 0.4628

comparison = pd.DataFrame({
    "Model": [
        "GRU + Multi-Head Attention",
        "LightGBM"
    ],
    "MAE": [
        gru_mae,
        lgb_test_mae
    ],
    "RMSE": [
        gru_rmse,
        lgb_test_rmse
    ],
    "R2": [
        gru_r2,
        lgb_test_r2
    ]
})

print("=" * 60)
print("GRU vs LIGHTGBM")
print("=" * 60)

display(comparison)

GRU vs LIGHTGBM


,Model,MAE,RMSE,R2
0,GRU + Multi-Head Attention,7.017800,20.785100,0.4628
1,LightGBM,30.830392,34.336106,-0.4661


In [15]:
# ============================================================
# Cell 13: Save LightGBM model and results
# ============================================================

import os
import joblib

os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Save model
MODEL_PATH = "models/lightgbm_inflation_model.pkl"
joblib.dump(lgbm_model, MODEL_PATH)

# Save comparison results
comparison.to_csv(
    "outputs/gru_vs_lightgbm_comparison.csv",
    index=False
)

# Save test predictions
lightgbm_predictions = test_df[
    ["Country Name", "Country Code", "Target_Year", "Target"]
].copy()

lightgbm_predictions["Predicted"] = lgb_test_pred
lightgbm_predictions["Absolute_Error"] = (
    lightgbm_predictions["Target"]
    - lightgbm_predictions["Predicted"]
).abs()

lightgbm_predictions.to_csv(
    "outputs/lightgbm_test_predictions.csv",
    index=False
)

print("=" * 60)
print("LIGHTGBM MODEL SAVED")
print("=" * 60)

print("Model:")
print(MODEL_PATH)

print("\nComparison:")
print("outputs/gru_vs_lightgbm_comparison.csv")

print("\nPredictions:")
print("outputs/lightgbm_test_predictions.csv")

LIGHTGBM MODEL SAVED
Model:
models/lightgbm_inflation_model.pkl

Comparison:
outputs/gru_vs_lightgbm_comparison.csv

Predictions:
outputs/lightgbm_test_predictions.csv
